# QF 627 Programming and Computational Finance
### Lesson 9 | Natural Language Processing (NLP) and Trading Strategies

> "Hi Team! 👋

> Today, we will learn how to capitalize on `natural language processing` (`NLP`) and `machine learning` to create `trading strategies`. As you will see, this will be informative, helpful knowledge and expertise for creating your trading strategy down the line. 

> First, we will define our real-world questions so that we have them in hand, and then we will prepare stock and news data. Next, we will learn how to go over sentiment analysis, along with supervised and unsupervised learning. Finally, we will go through an assessment of our model and trading strategy on individual stocks and multiple stocks over varying time periods.

### Learning Pointers 

### [Data Import and Transformation](#wrangle)
<br>

### [Natural Language Processing, Sentiment Analysis, and Machine Learning](#nlp)
<br>

### [Predictive Model Assessment, Strategy Building, and Backtesting](#test)

### Activation of necessary modules

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

from pandas_datareader import data as pdr

import datetime as dt
import yfinance as yf

#### Setting plotting and display options

In [ ]:
np.set_printoptions(precision = 3)

pd.set_option("display.float_format", lambda x: "%.3f" % x)

plt.style.use("ggplot")

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

In [ ]:
%matplotlib inline

    PROBLEM STATEMENT
    
> `The real-world question` in our hand is to create a trading strategy that capitalizes on natural language processing to extract information from news headlines, assign a sentiment to that information, and use the sentiment and the information inside the news headlines to create a trading strategy.

We will employ data from three different sources:

* `Yahoo Finance` website for the stock return: The return data can be obtained from another website that is similar to Yahoo finance.
<br>

* `News headlines` data compiled from the RSS feeds of several news websites: This news headlines data is compiled by various news websites and contains the most financially relevant news filtered by human editors. For this study, we will only look at the headline, not the details in the text of the story. Another important characteristic of this dataset is that the relevant tickers in the story are extracted. Our dataset contains 82,643 headlines from May 2011 through December 2018.
<br>

* `Kaggle`: Labeled data of news sentiments obtained for a classification-based sentiment analysis model. This data may not be authentic and is used only for demonstration purposes in our real-world problem solving.
<br>

* A `stock market lexicon` created based on stock market conversations in microblogging services.

### Load additional nececessary packages for our work

In [ ]:
!pip install spacy
!pip install nltk
!pip install textblob

!pip install backtrader

#### NLP libraries

In [ ]:
from textblob import TextBlob

import spacy

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download("vader_lexicon")

import warnings

import csv

# https://spacy.io/usage/models

In [ ]:
!pip install -U pip setuptools wheel
!pip install -U spacy
!python -m spacy download en_core_web_sm

In [ ]:
import en_core_web_sm
nlp = en_core_web_sm.load()

#### Libraries for processing news headlines

In [ ]:
from lxml import etree

import json

from io import StringIO
from os import listdir
from os.path import isfile, join
from copy import copy

from pandas.tseries.offsets import BDay
from scipy.stats.mstats import winsorize

#### Libraries for Classification for modeling the sentiments

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

### [NOTE: How to Install TensorFlow GPU for Mac M1/M2 with Conda--A Tutorial YouTube Video](https://www.youtube.com/watch?v=5DgWvU0p2bk)

#### Keras package for the deep learning model for the sentiment prediction. 

In [ ]:
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences

from keras.models import Sequential
from keras.layers import Dense, Flatten, LSTM, Dropout, Activation
from keras.layers import Embedding

#### Some additional libraries

In [ ]:
import statsmodels.api as sm
import seaborn as sns
import datetime
from datetime import date

import json  
import zipfile
import os.path
import sys

In [ ]:
# Diable the warnings
import warnings
warnings.filterwarnings("ignore")

<a id="wrangle"></a>

### Data Import and Transformation

#### Stock Data

In [ ]:
# tickers = ["AAPL",
#            "MSFT",
#            "AMZN",
#            "GOOG",
#            "META",
#            "WMT",
#            "JPM",
#            "TSLA",
#            "NFLX",
#            "ADBE"]

# start = "2010-01-01"
# end = "2018-12-31"

In [ ]:
# DF_stock_return = pd.DataFrame()

# for ticker in tickers:
#     ticker_YF = yf.Ticker(ticker)
    
#     if DF_stock_return.empty:
#         DF_stock_return = ticker_YF.history(start = start,
#                                             end = end)
#         DF_stock_return["ticker"] = ticker

#     else:
#         temporary_data = ticker_YF.history(start = start,
#                                            end = end)
#         temporary_data["ticker"] = ticker
#         DF_stock_return = pd.concat([DF_stock_return, 
#                                      temporary_data]
#                                    )

# DF_stock_return

In [ ]:
# DF_stock_return.to_csv("stocks.csv")

> The next step of data preparation is comprised of the following two overarching steps:

* Loading and pre-processing the news data

* Preparing the combined data


#### Processing News Data

> The news data is downloaded from the news RSS feed, the file is downloaded in JSON format, and the JSON files for different dates are kept in a zipped folder. Let us look at the contents of the JSON file.

In [ ]:
# headlines = zipfile.ZipFile("headlines.zip", "r")

# Test = headlines.namelist()[10]

# DataFile = headlines.open(Test).read()

# DataFile_Sample = json.loads(DataFile)["content"][1:500]
# DataFile_Sample

> As we can see, the JSON format is not suitable for the algorithm. We need to get the news out of the JSONs, using the following function. Regex is the vital part of this step. Regex can find a pattern in the raw, messy text and perform actions accordingly.

In [ ]:
# def ParseJSON(json_data):
#     XML = json_data["content"]
    
#     TREE = etree.parse(StringIO(XML),
#                        parser = etree.HTMLParser()
#                       )
    
#     HEADLINES = TREE.xpath("//h4[contains(@class, 'media-heading')]/a/text()")
#     assert len(HEADLINES) == json_data["count"]
    
#     MAIN_TICKERS = list(map(lambda x: x.replace("/symbol/", ""),
#                             TREE.xpath("//div[contains(@class, 'media-left')]//a/@href")
#                            )
#                        )
#     assert len(MAIN_TICKERS) == json_data["count"]
    
#     FINAL_HEADLINES = ["".join(f.xpath('.//text()')
#                               ) for f in TREE.xpath("//div[contains(@class, 'media-body')]/ul/li[i]")
#                       ]
#     if len(FINAL_HEADLINES) == 0:
#         FINAL_HEADLINES = ["".join(f.xpath('.//text()')
#                                   ) for f in TREE.xpath("//div[contains(@class, 'media-body')]"
#                                                        )
#                           ]
#         FINAL_HEADLINES = [f.replace(h, '').split('\xa0')[0].strip() for f, h in zip(FINAL_HEADLINES, HEADLINES)
#                           ]
#     return MAIN_TICKERS, FINAL_HEADLINES

In [ ]:
# ParseJSON(json.loads(DataFile))[1][1]

In [ ]:
# Data = None

# Data_News = []

# Ret = []

# Ret_F = []

# with zipfile.ZipFile("headlines.zip", "r") as z:
#     for filename in z.namelist():
        
#         try:
#             with z.open(filename) as f:
#                 Data = f.read()
#                 json_data = json.loads(Data)
                
#             if json_data.get("count", 0) > 10:
                
#                 MAIN_TICKERS, FINAL_HEADLINES = ParseJSON(json_data)
#                 if len(FINAL_HEADLINES) != json_data["count"]:
#                     continue
            
#                 FILE_DATE = filename.split("/")[-1].replace(".json", "")
#                 FILE_DATE = date(int(FILE_DATE[:4]), int(FILE_DATE[5:7]), int(FILE_DATE[8:])
#                                  )
                
#                 DF_Dict = {"ticker": MAIN_TICKERS,
#                            "headlines": FINAL_HEADLINES,
#                            "date": [FILE_DATE] * len(MAIN_TICKERS)
#                           }
                
#                 DF_F = pd.DataFrame(DF_Dict)
#                 Data_News.append(DF_F)
        
#         except:
#             pass

In [ ]:
# Data_News = pd.concat(Data_News)
# Data_News

> We can use the JSON parser to extract the news headlines from the complex HTML format. This format is good enough to be used for further analysis.

> Next, we extract the ticker and the headlines from all the JSON files and put that data in a data frame.

> As we can see, the data includes the ticker, headlines, and the date that will be used in the next step for combining with the return.

#### Building combined dataset

> In this step, we extract the event return, which is the return that corresponds to the event. We do this because, at times, the news is reported late, and at other times, it is reported after market close. Having a slightly wider window ensures that we will be able to capture the essence of the event. Event return is defined as follows:

$$ R_{t-1} + R_t + R_{t+1} $$

where, $ R_{t-1} $, $ R_{t+1} $ are the return before and after the news data and $ R_{t} $ is the return on
the day of the news (namely, time ***t***)

In [ ]:
# DF_stock_return["Return"] = DF_stock_return["Close"].pct_change()

#### Event Return

In [ ]:
# DF_stock_return["Event_Return"] =\
#    DF_stock_return["Return"] + DF_stock_return["Return"].shift(-1) + DF_stock_return["Return"].shift(1)

In [ ]:
# DF_stock_return.reset_index(level = 0,
#                             inplace = True)

# DF_stock_return["date"] = pd.to_datetime(DF_stock_return["Date"]
#                                         ).apply(lambda x: x.date()
#                                                )

# DF_stock_return

> Now that we have all the data in place, we will prepare a combined data frame that will have the news headlines mapped to the date, event return, and stock ticker. 

> This data frame will be used for further analysis for the sentiment analysis model and to build the trading strategy.

In [ ]:
# Data_News

In [ ]:
# Combined_DF = pd.merge(Data_News, DF_stock_return,
#                        how = "left",
#                        left_on = ["date", "ticker"],
#                        right_on = ["date", "ticker"]
#                        )

# Combined_DF

In [ ]:
# Combined_DF = Combined_DF[Combined_DF["ticker"].isin(tickers)
#                          ]

# Combined_DF

In [ ]:
# Data_DF = Combined_DF[["ticker", "headlines", "date", "Event_Return", "Close"]]

# Data_DF = Data_DF.dropna()

# Data_DF

In [ ]:
# Data_DF.dropna().to_csv(r"NEWS_RETURN.csv", sep = "|", index = False)

In [ ]:
# Data_DF = pd.read_csv(r"NEWS_RETURN.csv", sep = "|")
# Data_DF = Data_DF.dropna()

> Let us save the data in a CSV file to be used later, so that the data processing step can be skipped when we are looking into analysis.

> In this step, we have prepared a clean data frame that has the ticker, headline, event return, return for a given day, and future return for 10 unique stock tickers with a total of 2759 rows of data.

<a id="nlp"></a>

### Natural Language Processing, Sentiment Analysis, and Machine Learning

> Now we will go through the process of following three different approaches to getting the sentiments for the news, which we will use to build the trading strategy.

* Predefined model-TextBlob package
<br>

* Tuned model-classification algorithms and LSTM
<br>

* Model based on the financial lexicon

> We will also explore the differences between various ways to perform the sentiment analysis. Let us go through the steps.

### `TextBlob` package

> The texblob sentiment function is a pre-trained model based on the `Naïve-Bayes classification` algorithm. Its purpose is to convert a sentence to a numerical value of sentiment between `-1` to `+1` and map adjectives frequently found in movie reviews to sentiment polarity scores, ranging from `-1` to `+1` (`negative`-`positive`), and a similar subjectivity score (`objective`-`subjective`). 

> We will apply this algorithm to all headline articles. Let us compute the sentiment for all the headlines in the data.

In [ ]:
# text_sample_1 =\
# "Bayer (OTCPK:BAYRY) started the week up 3.5% to €74/share in Frankfurt, touching their \
# highest level in 14 months, after the U.S. government said a $25M glyphosate decision against the \
# company should be reversed."

In [ ]:
# TextBlob(text_sample_1).sentiment.polarity

> The sentiment polarity is a number between -1 (Very Negative) and +1 (Very Positive). We apply this on all headlines we have in the data that were processed in the previous step. Let us compute the sentiment for all the headlines in the data.

In [ ]:
# Data_DF["sent_textblob"] = [TextBlob(s).sentiment.polarity for s in Data_DF["headlines"]
#                             ]

> Let us analyze the scatterplot of the sentiments and the return.

In [ ]:
# plt.scatter(Data_DF["sent_textblob"], 
#             Data_DF["Event_Return"],
#             alpha = 0.25)

# plt.title("Relationships between Event Return and Sentiments (for All Data)")

# plt.xlabel("Sentiment")
# plt.ylabel("Event_Return")

# plt.show()

In [ ]:
# corr = Data_DF["Event_Return"].corr(Data_DF["sent_textblob"]
#                                    )
# corr

In [ ]:
# Data_DF_MSFT = Data_DF[Data_DF["ticker"] == "MSFT"]

# plt.scatter(Data_DF_MSFT["sent_textblob"], Data_DF_MSFT["Event_Return"], alpha = 0.75)

# plt.title("Relationships between Event Return and Sentiments (for MSFT)")

# plt.xlabel("Sentiment")
# plt.ylabel("Event_Return")

# plt.show()

> Overall, from the results we can see that there isn't a strong correlation between the news and the sentiments. Also, there are a lot of sentiments centered around zero.

> We see that the statement has a positive sentiment of 0.5, but looking at the words that give rise to the sentiment score, we find that it is the word "touching" and not "high" that causes positive sentiment.

> In this step, we develop a customized model for sentiment analysis, based on available labeled data. The labeled data for this is obtained from the kaggle website. Let’s look at the data.

> The data has headlines for the news across 30 different stocks, with a total of 9470 rows, and has sentiments labeled 0 and 1. The headlines are already almost cleaned. We perform the classification steps that we learned in the previous weeks.

In [ ]:
# text_sample_2 = "Bayer (OTCPK:BAYRY) started the week up 3.5% to €74/share in Frankfurt, touching their highest level in 14 months, after the U.S. government said a $25M glyphosate decision against the company should be reversed."

In [ ]:
# TextBlob(text_sample_2).sentiment_assessments

We see that the statement has a positive sentiment of .5 but looking at the words that give rise to the sentiments, the word "touching" and not "high" causes positive sentiment. 

### Supervised Machine Learning

> In order to run a supervised learning model, we first need to convert the news headlines into feature representation.

In [ ]:
# Sentiments_Data = pd.read_csv("labelled_news.csv",
#                               encoding = "ISO-8859-1")

# Sentiments_Data

> Now that we have prepared the independent variable, we will train the classification model in a similar manner to the technique learned in the previous lessons. 

> We will first divide the data into a training set and a test set and then run the key classification models.

> Let's run all the classification models.

#### Word-embedding

In [ ]:
# all_vectors = np.array([np.array([token.vector for token in nlp(s)]
#                                       ).mean(axis = 0) * np.ones((96)
#                                                                    ) for s in Sentiments_Data["headline"]
#                           ]
#                          )

In [ ]:
# Y = Sentiments_Data["sentiment"]
# X = all_vectors

In [ ]:
# from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV

In [ ]:
# val_size = 0.25

# seed = 7

# num_folds = 10

# scoring = "accuracy"

# X_Train, X_Test, Y_Train, Y_Test = train_test_split(X, Y,
#                                                     test_size = val_size,
#                                                     random_state = seed)

In [ ]:
# MODELS = []

In [ ]:
# MODELS.append(("LR", LogisticRegression()
#               )
#              )

# MODELS.append(("KNN", KNeighborsClassifier()
#               )
#              )

# MODELS.append(("SVM", SVC()
#               )
#              )

# MODELS.append(("CART", DecisionTreeClassifier()
#               )
#              )

# MODELS.append(("NN", MLPClassifier()
#               )
#              )

# MODELS.append(("RF", RandomForestClassifier()
#               )
#              )

In [ ]:
# results = []

# algorithms = []

# kfold_results = []

# train_results = []

# test_results = []

# for algo, MODEL in MODELS:
    
#     kfold = KFold(n_splits = num_folds,
#                   shuffle = True,
#                   random_state = seed)
    
#     cv_results = cross_val_score(MODEL,
#                                  X_Train,
#                                  Y_Train,
#                                  cv = kfold,
#                                  scoring = scoring)
    
#     results.append(cv_results)
    
#     algorithms.append(algo)
    
#     res = MODEL.fit(X_Train, Y_Train)
    
#     train_result = accuracy_score(res.predict(X_Train), Y_Train)
    
#     train_results.append(train_result)
    
#     test_result = accuracy_score(res.predict(X_Test), Y_Test)
    
#     test_results.append(test_result)

#     Message = "%s: %f (%f) %f %f" % (algo, cv_results.mean(), cv_results.std(), train_result, test_result)
    
#     print(Message)
    
#     print(confusion_matrix(res.predict(X_Test), Y_Test)
#          )

### Let's compare the Performance of Algorithms

In [ ]:
# fig = plt.figure()

# ind = np.arange(len(algorithms)
#                )

# width = 0.30

# fig.suptitle("Comparing Algorithms (Accuracy)")

# ax = fig.add_subplot(111)

# plt.bar(ind - width/2, train_results, 
#         width = width,
#         label = "Accuracy in TRAIN Set")

# plt.bar(ind + width/2, test_results, 
#         width = width,
#         label = "Accuracy in TEST Set")

# plt.legend()

# ax.set_xticks(ind)
# ax.set_xticklabels(algorithms)

# plt.show()

### LSTM based model

***If I could see you in one more class, we will go for this :)***

### Unsupervised Machine Learning with Fiancial Lexicon

> Lexicons are special dictionaries or vocabularies that have been created for analyzing sentiments.
VADER (Valence Aware Dictionary for Entiment Reasoning) is a pre-built sentiment analysis model included in the NLTK package. 

In [ ]:
# SIA = SentimentIntensityAnalyzer()

In [ ]:
# stock_lexicon = pd.read_csv("lexicon.csv")

# stock_lexicon

In [ ]:
# stock_lexicon["sentiment"] = (stock_lexicon["Aff_Score"] + stock_lexicon["Neg_Score"]
#                              )/2

In [ ]:
# stock_lexicon = dict(zip(stock_lexicon.Item, 
#                          stock_lexicon.sentiment)
#                     )

# stock_lexicon = {k:v for k, v in stock_lexicon.items() if len(k.split(" "))==1}

# stock_lexicon_scaled = {}

# for k, v in stock_lexicon.items():
#     if v > 0:
#         stock_lexicon_scaled[k] = v / max(stock_lexicon.values()
#                                          ) * 4
        
#     else:
#         stock_lexicon_scaled[k] = v / min(stock_lexicon.values()
#                                          ) * -4
        
# final_lexicon = {}
# final_lexicon.update(stock_lexicon_scaled)
# final_lexicon.update(SIA.lexicon)
# SIA.lexicon = final_lexicon

In [ ]:
# text_sample_3 = "AAPL has little competition from any of the tech companies"

In [ ]:
# text_sample_3 = "AAPL is trading higher after reporting its October sales rose 11.2% M/M. It has seen a 25% jump in orders"

In [ ]:
# SIA.polarity_scores(text_sample_3)["compound"]

In [ ]:
# TextBlob(text_sample_3).sentiment_assessments

In [ ]:
# vader_sentiments = np.array([SIA.polarity_scores(s)["compound"] for s in Data_DF["headlines"]
#                                ]
#                               )
# vader_sentiments.shape

In [ ]:
Data_DF

In [ ]:
# Data_DF["sent_lex"] = vader_sentiments

In [ ]:
# corr = Data_DF["Event_Return"].corr(Data_DF["sent_lex"]
#                                    )
# corr

In [ ]:
# plt.scatter(Data_DF["sent_lex"], Data_DF["Event_Return"],
#            alpha = 0.25)

# plt.ylabel("Event Return")
# plt.xlabel("Sentiments")

In [ ]:
# Data_DF_Amazon = Data_DF[Data_DF["ticker"] == "AMZN"]
# plt.scatter(Data_DF_Amazon["sent_lex"], Data_DF_Amazon["Event_Return"],
#            alpha = 0.25)

# plt.ylabel("Event Return")
# plt.xlabel("Sentiments")

### EDA and Comparisons

> Let us look at the sample headlines and the sentiments from three different methodologies, followed by an analysis that makes use of visualization.

In [ ]:
# New_Data_DF = Data_DF[Data_DF["ticker"] == "NFLX"][["ticker", "headlines", "sent_textblob", "sent_lex"]]

# from pandas import option_context

# with option_context("display.max_colwidth", 400):
#     display(New_Data_DF)

In [ ]:
# text = 'LinkedIn (LNKD) could have an exceptional drop in its price over the coming months'

In [ ]:
# data = [["LNKD", "LinkedIn (LNKD) could have an exceptional drop in its price over the coming months", -.1945, 0.66]]

# DF = pd.DataFrame(data, columns = ["ticker", "News", "sentiment_financial", "sentiment_mov"])

# with option_context("display.max_colwidth", 200):
#     display(DF)

In [ ]:
# SIA.polarity_scores(text)["compound"]

In [ ]:
# TextBlob(text).sentiment.polarity

In [ ]:
# TextBlob(text).sentiment_assessments.assessments

In [ ]:
# corr = Data_DF[["sent_textblob", "sent_lex", "Event_Return"]].dropna(axis =0).corr()

# plt.figure(figsize = (8, 8)
#           )
# plt.title("Correlation Matrix")
# sns.heatmap(corr[["Event_Return"]],
#             annot = True)

In [ ]:
# corr_data = []

# for ticker in Data_DF["ticker"].unique():

#     New_Data_DF = Data_DF[Data_DF["ticker"] == ticker]
    
#     if New_Data_DF.shape[0] > 40 :
#         corr_textblob = New_Data_DF["Event_Return"].corr(New_Data_DF["sent_textblob"])
#         corr_lex = New_Data_DF["Event_Return"].corr(New_Data_DF["sent_lex"])
#         corr_data.append([ticker, corr_textblob, corr_lex])
#     else:
#         continue

In [ ]:
# corr_DF = pd.DataFrame(corr_data,
#                        columns = ["ticker", "corr_textblob", "corr_lex"]
#                        )
# corr_DF = corr_DF.set_index("ticker")
# corr_DF

<a id="test"></a>

### [Predictive Model Assessment, Strategy Building, and Backtesting](#test)

> Sentiment data can be used in different ways for the trading strategy. Sentiment scores can be used as a directional signal and ideally create a long-short portfolio, by buying the stocks with a positive score and selling the stocks with a negative score. 

> The sentiments can also be used as additional features over and above other features (such as correlated stocks and technical indicators) in a supervised learning model to predict the price or come up with a trading strategy.

> In the trading strategy in the current problem solving task, we buy and sell stock as per the current stock sentiments: 

* Buy a stock when the change in sentiment score (current sentiment score - previous sentiment score) is greater than 0.5 and sell a stock when the change in sentiment score is less than -0.5.
<br>

* In addition, we check the 15-day moving average when buying or selling in a unit of 100.

> Surely, there can be many ways to create a trading strategy based on sentiments, by varying the threshold or changing the number of units based on the initial cash available.

> We will use lexicon-based sentiments for the trading strategy.

### Our Strategy

In [ ]:
# import backtrader as bt
# import backtrader.indicators as btind
# import backtrader.analyzers as btanalyzers

> Here we will use Backtrader, a Python-based API for writing and backtesting trading strategy.  

> Backtrader allows you to focus on writing reusable trading strategies, indicators, and analyzers instead of having to spend time building infrastructure. We have a convenient framework to backtest and write our trading strategy. 

> We will implement a simple strategy to buy if the previous day’s sentiment score increases by 0.5 from the last day and sell if it decreases by 0.5.

The following function contains two classes:

* Sentiment

* SentimentStrat: The "next" function of this class implements the actual trading strategy.


In [ ]:
# class Sentiment(bt.Indicator):
#     lines = ("sentiment",)
#     plotinfo = dict(
#         plotymargin=0.5,
#         plothlines=[0],
#         plotyticks=[1.0, 0, -1.0])
    
#     def next(self):
#         self.sentiment = 0.0
#         self.date = self.data.datetime
#         date = bt.num2date(self.date[0]
#                           ).date()
#         prev_sentiment = self.sentiment        
#         if date in date_sentiment:
#             self.sentiment = date_sentiment[date]
#         self.lines.sentiment[0] = self.sentiment

In [ ]:
# class SentimentStrat(bt.Strategy):
#     params = (
#         ("period", 15),
#         ("printlog", True),
#     )

#     def log(self, txt, dt=None, doprint=False):
#         """ 
#         Logging function for this strategy
#         """
        
#         if self.params.printlog or doprint:
#             dt = dt or self.datas[0].datetime.date(0)
#             print("%s, %s" % (dt.isoformat(), txt)
#                  )

#     def __init__(self):
        
#         # Keep a reference to the "close" line in the data[0] dataseries
        
#         self.dataclose = self.datas[0].close
        
#         # Keep track of pending orders
        
#         self.order = None
#         self.buyprice = None
#         self.buycomm = None
#         self.sma = bt.indicators.SimpleMovingAverage(self.datas[0], period=self.params.period)
#         self.date = self.data.datetime
#         self.sentiment = None
#         Sentiment(self.data)
#         self.plotinfo.plot = False
        
        
#     def notify_order(self, order):
#         if order.status in [order.Submitted, order.Accepted]:
            
#             # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            
#             return
        
#         # Check if an order has been completed
        
#         ###### ATTENTION: broker could reject order if not enough cash #######
        
#         if order.status in [order.Completed]:
#             if order.isbuy():
#                 self.log(
#                     "BUY DONE, Price: %.2f, Cost: %.2f, Comm %.2f" %
#                     (order.executed.price,
#                      order.executed.value,
#                      order.executed.comm))
#                 self.buyprice = order.executed.price
#                 self.buycomm = order.executed.comm
#             else:  # Sell
#                 self.log("SELL DONE, Price: %.2f, Cost: %.2f, Comm %.2f" %
#                          (order.executed.price,
#                           order.executed.value,
#                           order.executed.comm))
                
#             self.bar_executed = len(self)     
            
#         elif order.status in [order.Canceled, order.Margin, order.Rejected]:
#             self.log("Order Canceled/Margin/Rejected")
            
#         # Write down: no pending order
        
#         self.order = None
        
#     def notify_trade(self, trade):
#         if not trade.isclosed:
#             return

#         self.log("Profit of the operation, GROSS %.2f, NET %.2f" %
#                  (trade.pnl, trade.pnlcomm))
    
#     #### Main Strategy ####
    
#     def next(self):        
#         date = bt.num2date(self.date[0]).date()
#         prev_sentiment = self.sentiment
#         if date in date_sentiment:
#             self.sentiment = date_sentiment[date]
        
#         # Check if an order is pending. if yes, we cannot send a 2nd one
        
#         if self.order:
#             return       
        
#         # If not in the market and previous sentiment not none
        
#         if not self.position and prev_sentiment:
            
#             # buy if current close more than sma AND sentiment increased by >= 0.5
            
#             if self.dataclose[0] > self.sma[0] and self.sentiment - prev_sentiment >= 0.5:
                
#                 self.log("Previous Sentiment %.2f, New Sentiment %.2f BUY CREATE, %.2f" % (prev_sentiment, self.sentiment, self.dataclose[0]
#                                                                                           )
#                         )    
                
#                 self.order = self.buy()
                
#         # Already in the market and previous sentiment not none
        
#         elif prev_sentiment:
        
#             # sell if current close less than sma AND sentiment decreased by >= 0.5
#             if self.dataclose[0] < self.sma[0] and self.sentiment - prev_sentiment <= -0.5:
#                 self.log("Previous Sentiment %.2f, New Sentiment %.2f SELL CREATE, %.2f" % (prev_sentiment, self.sentiment, self.dataclose[0]
#                                                                                            )
#                         )            
                
#                 self.order = self.sell()

#     def stop(self):
#         self.log("(MA Period %2d) Ending Value %.2f" %
#                  (self.params.period, self.broker.getvalue()), doprint=True)
        
        

### Function for running the trading strategy

> Now, let’s build a generic to run the strategy for any stock. 

> We will specify the “ticker” stock feeds to be pulled from Yahoo Finance, and set an initial amount of $100,000, with a fixed size of 100 lots per trade.

In [ ]:
# def run_our_strategy(ticker, start, end):
#     print(ticker)
    
#     ticker = yf.Ticker(ticker)
    
#     df_ticker = ticker.history(start = start,
#                                end = end)
    
#     BRAIN = bt.Cerebro()
    
#     BRAIN.addstrategy(SentimentStrat)
    
#     data = bt.feeds.PandasData(dataname = df_ticker)
    
#     BRAIN.adddata(data)
    
#     start = 100000.0
    
#     BRAIN.broker.setcash(start)
    
#     BRAIN.addsizer(bt.sizers.FixedSize, stake = 100)
    
#     print("Starting Value of Our Portfolio: %.2f" % start)
    
#     plt.rcParams["font.size"] = "14"
    
#     plt.rcParams["figure.figsize"] = [14, 10]
    
#     BRAIN.run()
    
#     BRAIN.plot(volume = False,
#                iplot = True,
#                plotname = ticker)
    
#     end = BRAIN.broker.getvalue()
    
#     print("Starting Value of Our Portfolio: %.2f\nFinal Value of Our Portfolio: %.2f\nProfit: %.2f\n" % (start, end, end - start)
#          )
    
#     return float(df_ticker["Close"][0]), (end - start)

### Individual Stocks

In [ ]:
# ticker = "GOOG"

# date_sentiment = Data_DF[Data_DF["ticker"].isin([ticker]
#                                                )
#                         ]

# date_sentiment = date_sentiment[["date", "sent_lex"]]

# date_sentiment["date"] = pd.to_datetime(date_sentiment["date"], 
#                                         format = "%Y-%m-%d").dt.date

# date_sentiment = date_sentiment.set_index("date")["sent_lex"]

# date_sentiment = date_sentiment.to_dict()

# run_our_strategy(ticker, 
#                  start = "2012-01-01",
#                  end = "2018-12-12")

In [ ]:
# Data_DF_GOOG = Data_DF[Data_DF["ticker"].isin([ticker])]

# previous_news = list(Data_DF_GOOG[Data_DF_GOOG["date"] == "2017-04-27"]["headlines"])
 
# present_news = list(Data_DF_GOOG[Data_DF_GOOG["date"] == "2017-04-28"]["headlines"])

# print("Previous News:", previous_news,"\n\n\n", "Present News:", present_news)

> First running the strategy for Google.

> The chart will divided into three panels.

* Top Panel: The top panel is the Cash Value Observer, which, as the name implies, keeps track of the cash and total portfolio value during the life of the backtesting run. As we can see in this panel, we started with $100,000.00 and the final value at the end is __________ .
<br>

* Second Panel: This panel is Trade Observer, which shows, at the end of a trade, the actual profit and loss. A trade is defined as opening a position and taking the position back to zero (directly or crossing over from long to short or short to long).
<br>

* Third Panel: This panel is Buy Sell Observer, which plots (in addition to the prices) where buy and sell operations have taken place. 
<br>

* Bottom Panel : This panel shows the sentiment score.

In [ ]:
# ticker = "AMZN"

# date_sentiment = Data_DF[Data_DF["ticker"].isin([ticker]
#                                                )
#                         ]

# date_sentiment = date_sentiment[["date", "sent_lex"]]

# date_sentiment["date"] = pd.to_datetime(date_sentiment["date"], 
#                                         format = "%Y-%m-%d").dt.date

# date_sentiment = date_sentiment.set_index("date")["sent_lex"]

# date_sentiment = date_sentiment.to_dict()

# run_our_strategy(ticker, 
#                  start = "2012-01-01",
#                  end = "2018-12-12")

### Multiple Stocks

In [ ]:
# results_tickers = {}

# for ticker in tickers:

#     date_sentiment = Data_DF[Data_DF["ticker"].isin([ticker]
#                                                )
#                         ]

#     date_sentiment = date_sentiment[["date", "sent_lex"]]


#     date_sentiment["date"] = pd.to_datetime(date_sentiment["date"], 
#                                         format = "%Y-%m-%d").dt.date


#     date_sentiment = date_sentiment.set_index("date")["sent_lex"]


#     date_sentiment = date_sentiment.to_dict()

#     results_tickers[ticker] = run_our_strategy(ticker, 
#                                                start = "2012-01-01",
#                                                end = "2018-12-12")

In [ ]:
# pd.DataFrame.from_dict(results_tickers).set_index([pd.Index(["Per Unit Start Price",
#                                                              "Strategy Profit"]
#                                                            )
#                                                   ]
#                                                  )

### Varying the strategy time period

In [ ]:
# results_tickers = {}

# for ticker in tickers:

#     date_sentiment = Data_DF[Data_DF["ticker"].isin([ticker]
#                                                )
#                         ]

#     date_sentiment = date_sentiment[["date", "sent_lex"]]


#     date_sentiment["date"] = pd.to_datetime(date_sentiment["date"], 
#                                         format = "%Y-%m-%d").dt.date


#     date_sentiment = date_sentiment.set_index("date")["sent_lex"]


#     date_sentiment = date_sentiment.to_dict()

#     results_tickers[ticker] = run_our_strategy(ticker, 
#                                                start = "2012-01-01",
#                                                end = "2015-05-04")

In [ ]:
# pd.DataFrame.from_dict(results_tickers).set_index([pd.Index(["Per Unit Start Price",
#                                                              "Strategy Profit"]
#                                                            )
#                                                   ]
#                                                  )

### What We've Learned and What's Next 

> We found a good performance of the sentiment-based strategy across all the stocks except _______. 

> Our sentiment-based strategy performs quite well in different time periods. The strategy can be further tweaked to modify the threshold and order size. 

> Additional metrics such as the Sharpe ratio and maximum drawdown can also be used to understand the performance of the strategy. This will be examined in Exercise Problem Set #5. Sentiments can also be used along with other features, such as correlated variables and technical indicators, for prediction.

> We performed a comparison of the models and concluded that one of the most important steps in training the model for sentiment analysis is training it using domain-specific vocabulary.

> We further used the sentiments as signals to develop different trading strategies. This initial result suggests that a model trained on financial lexicon-based sentiments could prove viable as a trading strategy.

> Additional improvements to the entire processes of NLP-based strategy building can be made by using more complex pre-trained sentiment analysis models, such as Bert by Google or different pre-trained NLP models available on open-source platforms. Existing NLP libraries fill in some of the pre-processing and encoding steps to allow us to focus on the inference step.

<!-- ![gift](https://images.squarespace-cdn.com/content/v1/53f3eb3ce4b077de0318f4ea/628aea48-e976-49e7-a779-f8355fb7e171/For_QF627.gif?format=2500w "gift") -->

> `Thank you for working with the script, Team 👍`